In [1]:
import requests
import pandas as pd
import time
import os
from tqdm import tqdm

In [2]:
with open("../data/raw/app_ids_large_seed.txt", "r") as file:
    app_ids = [int(line.strip()) for line in file if line.strip()]

print(len(app_ids))
print(app_ids[:10])

483
[10, 20, 30, 40, 50, 60, 70, 80, 130, 220]


In [3]:
def get_current_players(app_id):
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={app_id}"
    
    try:
        response = requests.get(url, timeout=10)
        data = response.json()
        
        return {
            "app_id": app_id,
            "current_players": data.get("response", {}).get("player_count")
        }
    
    except Exception as e:
        print(f"Error for app_id {app_id}: {e}")
        return {
            "app_id": app_id,
            "current_players": None
        }

In [4]:
engagement_records = []

for app_id in tqdm(app_ids, desc="Collecting engagement data"):
    row = get_current_players(app_id)
    engagement_records.append(row)
    time.sleep(1)

df_engagement = pd.DataFrame(engagement_records)
df_engagement.head()

,app_id,current_players
0,10,13097.0
1,20,48.0
2,30,89.0
3,40,4.0
4,50,109.0


In [5]:
print(df_engagement.shape)
print(df_engagement.info())
print(df_engagement.isnull().sum())
print(df_engagement)

(483, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 483 entries, 0 to 482
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   app_id           483 non-null    int64  
 1   current_players  482 non-null    float64
dtypes: float64(1), int64(1)
memory usage: 7.7 KB
None
app_id             0
current_players    1
dtype: int64
      app_id  current_players
0         10          13097.0
1         20             48.0
2         30             89.0
3         40              4.0
4         50            109.0
..       ...              ...
478  2694490           9115.0
479  2708800              0.0
480  2767030          70523.0
481  2784840            117.0
482  2807960          41886.0

[483 rows x 2 columns]


In [6]:
os.makedirs("../data/raw", exist_ok=True)
df_engagement.to_csv("../data/raw/steam_engagement_batch1.csv", index=False)

print("Saved raw engagement batch.")

Saved raw engagement batch.


In [7]:
df_engagement_clean = df_engagement.copy()

df_engagement_clean.columns = df_engagement_clean.columns.str.strip().str.lower()
df_engagement_clean = df_engagement_clean.drop_duplicates(subset="app_id")
df_engagement_clean["current_players"] = pd.to_numeric(df_engagement_clean["current_players"], errors="coerce")

os.makedirs("../data/cleaned", exist_ok=True)
df_engagement_clean.to_csv("../data/cleaned/steam_engagement_clean.csv", index=False)

print("Saved cleaned engagement data.")

Saved cleaned engagement data.
